In [1]:
from google.colab import drive
drive.mount('/content/drive')

!apt-get -qq install unrar > /dev/null
!mkdir -p /content/raabin_raw
!unrar x -idq /content/drive/MyDrive/WBC-Folder/raabin/Train.rar /content/raabin_raw/
!unrar x -idq /content/drive/MyDrive/WBC-Folder/raabin/TestA.rar /content/raabin_raw/

import os
root0 = "/content/raabin_raw"
for root, dirs, files in os.walk(root0):
    depth = root.count(os.sep) - root0.count(os.sep)
    if depth <= 2:
        print(root, "|", len(dirs), "folders |", len(files), "files |", files[:3])

Mounted at /content/drive

Cannot open /content/drive/MyDrive/WBC-Folder/raabin/Train.rar
No such file or directory
Cannot open /content/drive/MyDrive/WBC-Folder/raabin/TestA.rar
No such file or directory/content/raabin_raw | 0 folders | 0 files | []


In [2]:
!ls -la /content/drive/MyDrive/raabin

total 402783
drwx------ 2 root root      4096 Nov 14 16:19 .
drwx------ 5 root root      4096 Nov 14 16:36 ..
-rw------- 1 root root 120139942 Nov 14 16:18 TestA.rar
-rw------- 1 root root 292300976 Nov 14 16:17 Train.rar


In [3]:
!mkdir -p /content/raabin_raw
!unrar x -idq /content/drive/MyDrive/raabin/Train.rar /content/raabin_raw/
!unrar x -idq /content/drive/MyDrive/raabin/TestA.rar /content/raabin_raw/

import os
root0 = "/content/raabin_raw"
for root, dirs, files in os.walk(root0):
    depth = root.count(os.sep) - root0.count(os.sep)
    if depth <= 2:
        print(root, "|", len(dirs), "folders |", len(files), "files |", files[:3])



/content/raabin_raw | 2 folders | 0 files | []
/content/raabin_raw/Train | 5 folders | 0 files | []
/content/raabin_raw/Train/Neutrophil | 0 folders | 6231 files | ['95-8-20-2_79_2.jpg', '95-7-24-2_378_1.jpg', '95-8-12-5_36_1.jpg']
/content/raabin_raw/Train/Lymphocyte | 0 folders | 2427 files | ['95-5-4-1_169_3.jpg', '95-9-4-5_60_1.jpg', '95-8-10-1_385_1.jpg']
/content/raabin_raw/Train/Monocyte | 0 folders | 561 files | ['95-8-10-1_1291_1.jpg', '95-5-11-1_40_1.jpg', '95-5-4-1_92_2.jpg']
/content/raabin_raw/Train/Eosinophil | 0 folders | 744 files | ['95-8-27-1_54_2.jpg', '95-8-27-1_115_3.jpg', '95-8-27-1_226_3.jpg']
/content/raabin_raw/Train/Basophil | 0 folders | 212 files | ['20190604_111117_0.jpg', '20190531_120958_0.jpg', '20190531_115100_0.jpg']
/content/raabin_raw/TestA | 5 folders | 0 files | []
/content/raabin_raw/TestA/Neutrophil | 0 folders | 2660 files | ['95-8-27-1_222_7.jpg', '95-8-22-4_81_1.jpg', '95-8-22-3_172_1.jpg']
/content/raabin_raw/TestA/Lymphocyte | 0 folders | 10

In [4]:
import os
import pandas as pd
from PIL import Image

root = "/content/raabin_raw"
rows = []
for split in ["Train", "TestA"]:
    for cls in sorted(os.listdir(f"{root}/{split}")):
        for f in os.listdir(f"{root}/{split}/{cls}"):
            rows.append((split, cls, f))
df = pd.DataFrame(rows, columns=["split", "cls", "file"])
df["prefix"] = df["file"].str.split("_").str[0]

print(df.groupby("cls")["prefix"].nunique().rename("unique prefixes"))
for cls in sorted(df["cls"].unique()):
    a = set(df[(df.split == "Train") & (df.cls == cls)]["prefix"])
    b = set(df[(df.split == "TestA") & (df.cls == cls)]["prefix"])
    print(cls, "| train prefixes:", len(a), "| testA prefixes:", len(b), "| shared:", len(a & b))
print(df.groupby("cls")["prefix"].apply(lambda s: s.value_counts().head(3).to_dict()))

for cls in sorted(os.listdir(f"{root}/Train")):
    f = os.listdir(f"{root}/Train/{cls}")[0]
    im = Image.open(f"{root}/Train/{cls}/{f}")
    print(cls, im.size, im.mode)

cls
Basophil       4
Eosinophil    49
Lymphocyte    56
Monocyte      53
Neutrophil    56
Name: unique prefixes, dtype: int64
Basophil | train prefixes: 4 | testA prefixes: 4 | shared: 4
Eosinophil | train prefixes: 49 | testA prefixes: 2 | shared: 2
Lymphocyte | train prefixes: 37 | testA prefixes: 24 | shared: 5
Monocyte | train prefixes: 38 | testA prefixes: 16 | shared: 1
Neutrophil | train prefixes: 43 | testA prefixes: 19 | shared: 6
cls                  
Basophil    20190531     133.0
            20190526      72.0
            20190527      72.0
            95-8-27-1      NaN
            95-5-4-1       NaN
            95-8-10-1      NaN
            95-5-10-1      NaN
            95-8-8-2       NaN
            95-9-1-2       NaN
            95-8-6-1       NaN
Eosinophil  20190531       NaN
            20190526       NaN
            20190527       NaN
            95-8-27-1    701.0
            95-5-4-1      36.0
            95-8-10-1     28.0
            95-5-10-1      NaN
        

In [5]:
summary = []
for cls in sorted(df["cls"].unique()):
    d = df[df.cls == cls]
    tr = set(d[d.split == "Train"]["prefix"])
    te = set(d[d.split == "TestA"]["prefix"])
    top = d["prefix"].value_counts()
    summary.append({
        "class": cls, "images": len(d), "prefixes": d["prefix"].nunique(),
        "train_prefixes": len(tr), "testA_prefixes": len(te), "shared": len(tr & te),
        "testA_imgs_in_shared": int(d[(d.split == "TestA") & d["prefix"].isin(tr)].shape[0]),
        "top_prefix_share": round(top.iloc[0] / len(d), 2),
    })
print(pd.DataFrame(summary).to_string(index=False))
print("unique prefixes outside basophil:", df[df.cls != "Basophil"]["prefix"].nunique())

     class  images  prefixes  train_prefixes  testA_prefixes  shared  testA_imgs_in_shared  top_prefix_share
  Basophil     301         4               4               4       4                    89              0.44
Eosinophil    1066        49              49               2       2                   322              0.66
Lymphocyte    3461        56              37              24       5                    33              0.09
  Monocyte     795        53              38              16       1                     8              0.13
Neutrophil    8891        56              43              19       6                   349              0.09
unique prefixes outside basophil: 56


In [6]:
import numpy as np, pandas as pd, os
base = "/content/drive/MyDrive/WBC-Folder/"
root = "/content/raabin_raw"
df["path"] = root + "/" + df["split"] + "/" + df["cls"] + "/" + df["file"]
main = df[df.cls != "Basophil"]
classes4 = ["Eosinophil", "Lymphocyte", "Monocyte", "Neutrophil"]
prefixes = sorted(main["prefix"].unique())
print(len(prefixes), "prefixes")

chosen = None
for seed in range(200):
    perm = list(np.random.RandomState(seed).permutation(prefixes))
    adapt_p, test_p = set(perm[:14]), set(perm[14:])
    ok = all(
        main[main.cls == c]["prefix"].isin(adapt_p).sum() >= 40 and
        main[main.cls == c]["prefix"].isin(test_p).sum() >= 150
        for c in classes4)
    if ok:
        chosen = seed
        break
print("slide split seed:", chosen)

test_parts, adapt_parts = [], []
for c in classes4:
    d = main[main.cls == c]
    test_parts.append(d[d.prefix.isin(test_p)].sample(100, random_state=42))
    adapt_parts.append(d[d.prefix.isin(adapt_p)].sample(20, random_state=7))
baso = df[df.cls == "Basophil"]
b_adapt = baso.sample(20, random_state=7)
b_test = baso[~baso.path.isin(b_adapt.path)].sample(100, random_state=42)
test = pd.concat(test_parts + [b_test]).reset_index(drop=True)
adapt = pd.concat(adapt_parts + [b_adapt]).reset_index(drop=True)
test["independent"] = test.cls != "Basophil"

print(len(test), len(adapt))
print("test vs adapt image overlap:", len(set(test.path) & set(adapt.path)))
t4, a4 = test[test.independent], adapt[adapt.cls != "Basophil"]
print("test vs adapt prefix overlap (non-basophil):", len(set(t4.prefix) & set(a4.prefix)))
print(test.cls.value_counts().to_dict())
print(adapt.cls.value_counts().to_dict())

for obj, name in [(test, "raabin_frozen_test_500.csv"), (adapt, "raabin_adapt_100.csv")]:
    if os.path.exists(base + name):
        print(name, "already exists, not overwriting")
    else:
        obj[["path", "cls", "prefix"] + (["independent"] if "independent" in obj else [])].to_csv(base + name, index=False)
        print("saved", name)

56 prefixes
slide split seed: 0
500 100
test vs adapt image overlap: 0
test vs adapt prefix overlap (non-basophil): 0
{'Eosinophil': 100, 'Lymphocyte': 100, 'Monocyte': 100, 'Neutrophil': 100, 'Basophil': 100}
{'Eosinophil': 20, 'Lymphocyte': 20, 'Monocyte': 20, 'Neutrophil': 20, 'Basophil': 20}
saved raabin_frozen_test_500.csv
saved raabin_adapt_100.csv


In [7]:
import os, shutil
import pandas as pd

base = "/content/drive/MyDrive/WBC-Folder/"
dest_root = "/content/drive/MyDrive/raabin/images_used/"

test = pd.read_csv(base + "raabin_frozen_test_500.csv"); test["split"] = "test"
adapt = pd.read_csv(base + "raabin_adapt_100.csv"); adapt["split"] = "adapt"
want = pd.concat([test, adapt]).reset_index(drop=True)
want["label"] = want["cls"].str.lower()
want["local"] = dest_root + want["cls"] + "/" + want["path"].apply(os.path.basename)
assert want["local"].is_unique, "file names collide"

for src, dst in zip(want["path"], want["local"]):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
        shutil.copy(src, dst)

have = sum(os.path.exists(p) and os.path.getsize(p) > 0 for p in want["local"])
print("images saved:", have, "of", len(want))
print(want.groupby(["split", "label"]).size())

out = base + "raabin_want_600.csv"
if os.path.exists(out):
    print("list already exists, not overwriting")
else:
    want.to_csv(out, index=False)
    print("saved", out)

images saved: 600 of 600
split  label     
adapt  basophil       20
       eosinophil     20
       lymphocyte     20
       monocyte       20
       neutrophil     20
test   basophil      100
       eosinophil    100
       lymphocyte    100
       monocyte      100
       neutrophil    100
dtype: int64
saved /content/drive/MyDrive/WBC-Folder/raabin_want_600.csv


In [8]:
import os, torch, torch.nn as nn
import pandas as pd
from PIL import Image
from torchvision import models, transforms
from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/WBC-Folder/"
device = "cuda" if torch.cuda.is_available() else "cpu"
classes = ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']
print("device:", device)

tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def fresh_model():
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(classes))
    m.load_state_dict(torch.load(base + "resnet18_wbc_v2.pt", map_location=device))
    return m.to(device).eval()

model = fresh_model()
want = pd.read_csv(base + "raabin_want_600.csv")
test = want[want["split"] == "test"].reset_index(drop=True)
assert set(test["label"]) <= set(classes), "label names do not match"

preds = []
with torch.no_grad():
    for i in range(0, len(test), 32):
        x = torch.stack([tfm(Image.open(p).convert("RGB")) for p in test["local"].iloc[i:i + 32]]).to(device)
        preds += [classes[k] for k in model(x).argmax(1).cpu().tolist()]

test["pred"] = preds
test["correct"] = test["pred"] == test["label"]
print("primary accuracy (400 independent):", round(test[test.label != "basophil"]["correct"].mean(), 3))
print("basophil (100, flagged):", round(test[test.label == "basophil"]["correct"].mean(), 3))
print("recall by class:", test.groupby("label")["correct"].mean().round(3).to_dict())
print("predicted share:", test["pred"].value_counts(normalize=True).round(3).to_dict())

out = base + "raabin_baseline_v2_test500.csv"
if os.path.exists(out):
    print("file already exists, not overwriting")
else:
    test.to_csv(out, index=False)
    print("saved", out)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cpu
primary accuracy (400 independent): 0.502
basophil (100, flagged): 0.0
recall by class: {'basophil': 0.0, 'eosinophil': 0.73, 'lymphocyte': 0.54, 'monocyte': 0.63, 'neutrophil': 0.11}
predicted share: {'eosinophil': 0.332, 'monocyte': 0.286, 'platelet': 0.132, 'lymphocyte': 0.118, 'ig': 0.084, 'erythroblast': 0.026, 'neutrophil': 0.022}
saved /content/drive/MyDrive/WBC-Folder/raabin_baseline_v2_test500.csv


In [1]:
import io, zipfile, os
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/WBC-Folder/"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
classes = ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']

norm = transforms.Compose([transforms.ToTensor(),
                           transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
plain_tfm = transforms.Compose([transforms.Resize((224, 224)), norm])

def shrink_pad(img, s, size=224):
    inner = int(size * s)
    small = img.resize((inner, inner))
    arr = np.array(img)
    border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
    bg = tuple(int(v) for v in np.median(border, axis=0))
    canvas = Image.new("RGB", (size, size), bg)
    canvas.paste(small, ((size - inner) // 2, (size - inner) // 2))
    return canvas

def bg_color(arr):
    flat = arr.reshape(-1, 3).astype("float32")
    lum = flat.mean(axis=1)
    return np.median(flat[lum >= np.percentile(lum, 90)], axis=0)

def bg_match(img, ref_bg):
    arr = np.array(img).astype("float32")
    gain = ref_bg / np.maximum(bg_color(arr), 1)
    return Image.fromarray(np.clip(arr * gain, 0, 255).astype("uint8"))

with zipfile.ZipFile(base + "PBC_dataset_normal_DIB.zip") as z:
    hit = [n for n in z.namelist() if "BNE_56697" in n and "__MACOSX" not in n][0]
    refA = bg_color(np.array(Image.open(io.BytesIO(z.read(hit))).convert("RGB")))
print("background A:", refA.round(0))

def prep_plain(p): return plain_tfm(Image.open(p).convert("RGB"))
def prep_fix(p): return norm(shrink_pad(bg_match(Image.open(p).convert("RGB"), refA), 0.65))

want = pd.read_csv(base + "raabin_want_600.csv")
adapt = want[want["split"] == "adapt"].reset_index(drop=True)
y = torch.tensor([classes.index(l) for l in adapt["label"]])
Xs = {"plain": torch.stack([prep_plain(p) for p in adapt["local"]]),
      "fixes": torch.stack([prep_fix(p) for p in adapt["local"]])}
print(Xs["plain"].shape, adapt["label"].value_counts().to_dict())

rng = np.random.RandomState(0)
fold = np.zeros(len(adapt), dtype=int)
for lab in adapt["label"].unique():
    idx = np.where(adapt["label"].values == lab)[0]
    rng.shuffle(idx)
    fold[idx] = np.arange(len(idx)) % 5

def fresh_model():
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(classes))
    m.load_state_dict(torch.load(base + "resnet18_wbc_v2.pt", map_location=device))
    return m.to(device)

def train_eval(X, setting, tr, va, epochs=15, bs=16, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    m = fresh_model()
    for p in m.parameters():
        p.requires_grad = False
    fc_params = list(m.fc.parameters())
    for p in fc_params:
        p.requires_grad = True
    groups = [{"params": fc_params, "lr": 1e-3}]
    if setting == "B":
        l4 = list(m.layer4.parameters())
        for p in l4:
            p.requires_grad = True
        groups.append({"params": l4, "lr": 1e-4})
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss()
    m.eval()  # keeps BatchNorm statistics frozen
    for ep in range(epochs):
        perm = torch.randperm(len(tr))
        for i in range(0, len(tr), bs):
            b = tr[perm[i:i + bs]]
            xb, yb = X[b].to(device), y[b].to(device)
            if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[3])
            if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[2])
            loss = lossf(m(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return m(X[va].to(device)).argmax(1).cpu()

is4 = (adapt["label"] != "basophil").values
def report(name, pred):
    ok = (pred == y).numpy()
    print(f"{name}: all 5 classes {ok.mean():.3f} | 4 classes {ok[is4].mean():.3f}")

m0 = fresh_model().eval()
for pre in ["plain", "fixes"]:
    with torch.no_grad():
        p0 = m0(Xs[pre].to(device)).argmax(1).cpu()
    report(f"no adaptation, {pre}", p0)

for pre in ["plain", "fixes"]:
    for setting in ["A", "B"]:
        pred_all = torch.zeros(len(y), dtype=torch.long)
        for k in range(5):
            va = torch.tensor(np.where(fold == k)[0])
            tr = torch.tensor(np.where(fold != k)[0])
            pred_all[va] = train_eval(Xs[pre], setting, tr, va)
        report(f"CV {pre} + setting {setting}", pred_all)

Mounted at /content/drive
device: cuda
background A: [255. 228. 203.]
torch.Size([100, 3, 224, 224]) {'eosinophil': 20, 'lymphocyte': 20, 'monocyte': 20, 'neutrophil': 20, 'basophil': 20}
no adaptation, plain: all 5 classes 0.240 | 4 classes 0.300
no adaptation, fixes: all 5 classes 0.180 | 4 classes 0.225
CV plain + setting A: all 5 classes 0.750 | 4 classes 0.713
CV plain + setting B: all 5 classes 0.780 | 4 classes 0.725
CV fixes + setting A: all 5 classes 0.700 | 4 classes 0.662
CV fixes + setting B: all 5 classes 0.850 | 4 classes 0.812


In [2]:
torch.manual_seed(0); np.random.seed(0)
m = fresh_model()
for p in m.parameters():
    p.requires_grad = False
fc_params = list(m.fc.parameters())
l4 = list(m.layer4.parameters())
for p in fc_params + l4:
    p.requires_grad = True
opt = torch.optim.AdamW([{"params": fc_params, "lr": 1e-3},
                         {"params": l4, "lr": 1e-4}], weight_decay=1e-4)
lossf = nn.CrossEntropyLoss()
m.eval()  # keeps BatchNorm statistics frozen
X = Xs["fixes"]
all_idx = torch.arange(len(y))
for ep in range(15):
    perm = torch.randperm(len(all_idx))
    for i in range(0, len(all_idx), 16):
        b = all_idx[perm[i:i + 16]]
        xb, yb = X[b].to(device), y[b].to(device)
        if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[3])
        if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[2])
        loss = lossf(m(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
print("trained on", len(all_idx), "images")

test = want[want["split"] == "test"].reset_index(drop=True)
preds = []
with torch.no_grad():
    for i in range(0, len(test), 32):
        xb = torch.stack([prep_fix(p) for p in test["local"].iloc[i:i + 32]]).to(device)
        preds += [classes[k] for k in m(xb).argmax(1).cpu().tolist()]
test["pred"] = preds
test["correct"] = test["pred"] == test["label"]
print("primary accuracy (400 independent):", round(test[test.label != "basophil"]["correct"].mean(), 3))
print("basophil (100, flagged):", round(test[test.label == "basophil"]["correct"].mean(), 3))
print("recall by class:", test.groupby("label")["correct"].mean().round(3).to_dict())
print("predicted share:", test["pred"].value_counts(normalize=True).round(3).to_dict())

out = base + "raabin_adapted_fixesB_test500.csv"
wts = base + "resnet18_wbc_v2_raabin_fixesB.pt"
if os.path.exists(out) or os.path.exists(wts):
    print("files already exist, not overwriting")
else:
    test.to_csv(out, index=False)
    torch.save(m.state_dict(), wts)
    print("saved", out, "and", wts)

trained on 100 images
primary accuracy (400 independent): 0.762
basophil (100, flagged): 1.0
recall by class: {'basophil': 1.0, 'eosinophil': 0.83, 'lymphocyte': 0.76, 'monocyte': 0.63, 'neutrophil': 0.83}
predicted share: {'eosinophil': 0.266, 'basophil': 0.216, 'neutrophil': 0.194, 'monocyte': 0.166, 'lymphocyte': 0.158}
saved /content/drive/MyDrive/WBC-Folder/raabin_adapted_fixesB_test500.csv and /content/drive/MyDrive/WBC-Folder/resnet18_wbc_v2_raabin_fixesB.pt


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!apt-get -qq install unrar > /dev/null
!mkdir -p /content/raabin_raw
!unrar x -idq /content/drive/MyDrive/raabin/Train.rar /content/raabin_raw/
!unrar x -idq /content/drive/MyDrive/raabin/TestA.rar /content/raabin_raw/

import os
import numpy as np, pandas as pd
base = "/content/drive/MyDrive/WBC-Folder/"
root = "/content/raabin_raw"
rows = []
for split in ["Train", "TestA"]:
    for cls in sorted(os.listdir(f"{root}/{split}")):
        for f in os.listdir(f"{root}/{split}/{cls}"):
            rows.append((split, cls, f))
df = pd.DataFrame(rows, columns=["split", "cls", "file"])
df["prefix"] = df["file"].str.split("_").str[0]
df["path"] = root + "/" + df["split"] + "/" + df["cls"] + "/" + df["file"]
main = df[df.cls != "Basophil"]

prefixes = sorted(main["prefix"].unique())
perm = list(np.random.RandomState(0).permutation(prefixes))
adapt_p, test_p = set(perm[:14]), set(perm[14:])

used_test = pd.read_csv(base + "raabin_frozen_test_500.csv")
used_adapt = pd.read_csv(base + "raabin_adapt_100.csv")
ut = used_test[used_test.cls != "Basophil"]
ua = used_adapt[used_adapt.cls != "Basophil"]
print("saved test prefixes all on test side:", ut.prefix.isin(test_p).all(),
      "| saved adapt prefixes all on adapt side:", ua.prefix.isin(adapt_p).all())

fresh_test = main[main.prefix.isin(test_p) & ~main.path.isin(used_test.path)]
extra_adapt = main[main.prefix.isin(adapt_p) & ~main.path.isin(used_adapt.path)]
print("fresh test pool:", fresh_test.cls.value_counts().to_dict())
print("extra adaptation pool:", extra_adapt.cls.value_counts().to_dict())
big = main[main.cls == "Eosinophil"]["prefix"].value_counts().index[0]
print("largest eosinophil prefix:", big, "| on", "test side" if big in test_p else "adaptation side")

Mounted at /content/drive
saved test prefixes all on test side: True | saved adapt prefixes all on adapt side: True
fresh test pool: {'Neutrophil': 7296, 'Lymphocyte': 2468, 'Eosinophil': 874, 'Monocyte': 501}
extra adaptation pool: {'Neutrophil': 1475, 'Lymphocyte': 873, 'Monocyte': 174, 'Eosinophil': 72}
largest eosinophil prefix: 95-8-27-1 | on test side


In [2]:
import shutil

# Fresh frozen test: 200 per class, 4 independent classes
test2 = pd.concat([d.sample(200, random_state=42) for _, d in fresh_test.groupby("cls")]).reset_index(drop=True)

# Extra adaptation images: 20 per class from adaptation-side samples
extra = pd.concat([d.sample(20, random_state=11) for _, d in extra_adapt.groupby("cls")])
baso = df[df.cls == "Basophil"]
baso_free = baso[~baso.path.isin(used_test.path) & ~baso.path.isin(used_adapt.path)]
extra = pd.concat([extra, baso_free.sample(20, random_state=11)]).reset_index(drop=True)

cols = ["path", "cls", "prefix"]
adapt2 = pd.concat([used_adapt[cols], extra[cols]]).reset_index(drop=True)

# Safety checks
print(len(test2), len(adapt2))
print("test2 vs adapt2 image overlap:", len(set(test2.path) & set(adapt2.path)))
print("test2 vs earlier test image overlap:", len(set(test2.path) & set(used_test.path)))
a4 = adapt2[adapt2.cls != "Basophil"]
print("test2 vs adapt2 prefix overlap (non-basophil):", len(set(test2.prefix) & set(a4.prefix)))
print(test2.cls.value_counts().to_dict())
print(adapt2.cls.value_counts().to_dict())

# Save all 1,000 images to Drive and write one master list
dest_root = "/content/drive/MyDrive/raabin/images_used2/"
test2["split"] = "test2"; adapt2["split"] = "adapt2"
want2 = pd.concat([test2, adapt2]).reset_index(drop=True)
want2["label"] = want2["cls"].str.lower()
want2["local"] = dest_root + want2["cls"] + "/" + want2["path"].apply(os.path.basename)
assert want2["local"].is_unique, "file names collide"
for src, dst in zip(want2["path"], want2["local"]):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
        shutil.copy(src, dst)
print("images saved:", sum(os.path.exists(p) and os.path.getsize(p) > 0 for p in want2["local"]), "of", len(want2))

out = base + "raabin_want2_1000.csv"
if os.path.exists(out):
    print("list already exists, not overwriting")
else:
    want2.to_csv(out, index=False)
    print("saved", out)

800 200
test2 vs adapt2 image overlap: 0
test2 vs earlier test image overlap: 0
test2 vs adapt2 prefix overlap (non-basophil): 0
{'Eosinophil': 200, 'Lymphocyte': 200, 'Monocyte': 200, 'Neutrophil': 200}
{'Eosinophil': 40, 'Lymphocyte': 40, 'Monocyte': 40, 'Neutrophil': 40, 'Basophil': 40}
images saved: 1000 of 1000
saved /content/drive/MyDrive/WBC-Folder/raabin_want2_1000.csv


In [3]:
import io, zipfile, os
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/WBC-Folder/"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
classes = ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']

norm = transforms.Compose([transforms.ToTensor(),
                           transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
plain_tfm = transforms.Compose([transforms.Resize((224, 224)), norm])

def shrink_pad(img, s, size=224):
    inner = int(size * s)
    small = img.resize((inner, inner))
    arr = np.array(img)
    border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
    bg = tuple(int(v) for v in np.median(border, axis=0))
    canvas = Image.new("RGB", (size, size), bg)
    canvas.paste(small, ((size - inner) // 2, (size - inner) // 2))
    return canvas

def bg_color(arr):
    flat = arr.reshape(-1, 3).astype("float32")
    lum = flat.mean(axis=1)
    return np.median(flat[lum >= np.percentile(lum, 90)], axis=0)

def bg_match(img, ref_bg):
    arr = np.array(img).astype("float32")
    gain = ref_bg / np.maximum(bg_color(arr), 1)
    return Image.fromarray(np.clip(arr * gain, 0, 255).astype("uint8"))

with zipfile.ZipFile(base + "PBC_dataset_normal_DIB.zip") as z:
    hit = [n for n in z.namelist() if "BNE_56697" in n and "__MACOSX" not in n][0]
    refA = bg_color(np.array(Image.open(io.BytesIO(z.read(hit))).convert("RGB")))
print("background A:", refA.round(0))

def prep_plain(p): return plain_tfm(Image.open(p).convert("RGB"))
def prep_fix(p): return norm(shrink_pad(bg_match(Image.open(p).convert("RGB"), refA), 0.65))

def fresh_model(path=base + "resnet18_wbc_v2.pt"):
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(classes))
    m.load_state_dict(torch.load(path, map_location=device))
    return m.to(device).eval()

want2 = pd.read_csv(base + "raabin_want2_1000.csv")
adapt2 = want2[want2["split"] == "adapt2"].reset_index(drop=True)
test2 = want2[want2["split"] == "test2"].reset_index(drop=True)
y = torch.tensor([classes.index(l) for l in adapt2["label"]])
X = torch.stack([prep_fix(p) for p in adapt2["local"]])
print("adaptation images:", len(adapt2), "| fresh test images:", len(test2))

# train adapted-40: same recipe as before (fixes + setting B, seed 0, 15 epochs)
torch.manual_seed(0); np.random.seed(0)
m40 = fresh_model()
for p in m40.parameters():
    p.requires_grad = False
fc_params = list(m40.fc.parameters())
l4 = list(m40.layer4.parameters())
for p in fc_params + l4:
    p.requires_grad = True
opt = torch.optim.AdamW([{"params": fc_params, "lr": 1e-3},
                         {"params": l4, "lr": 1e-4}], weight_decay=1e-4)
lossf = nn.CrossEntropyLoss()
m40.eval()  # keeps BatchNorm statistics frozen
all_idx = torch.arange(len(y))
for ep in range(15):
    perm = torch.randperm(len(all_idx))
    for i in range(0, len(all_idx), 16):
        b = all_idx[perm[i:i + 16]]
        xb, yb = X[b].to(device), y[b].to(device)
        if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[3])
        if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[2])
        loss = lossf(m40(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
print("trained adapted-40 on", len(all_idx), "images")

Xt_plain = torch.stack([prep_plain(p) for p in test2["local"]])
Xt_fix = torch.stack([prep_fix(p) for p in test2["local"]])

def predict_t(m, Xt):
    m.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(Xt), 64):
            out += [classes[k] for k in m(Xt[i:i + 64].to(device)).argmax(1).cpu().tolist()]
    return np.array(out)

res = test2[["path", "label"]].copy()
res["plain"] = predict_t(fresh_model(), Xt_plain)
res["adapted20"] = predict_t(fresh_model(base + "resnet18_wbc_v2_raabin_fixesB.pt"), Xt_fix)
res["adapted40"] = predict_t(m40, Xt_fix)
for name in ["plain", "adapted20", "adapted40"]:
    ok = res[name] == res["label"]
    print(name, "accuracy:", round(ok.mean(), 3), "| recall:", ok.groupby(res["label"]).mean().round(3).to_dict())
b_ = ((res.adapted20 == res.label) & (res.adapted40 != res.label)).sum()
c_ = ((res.adapted20 != res.label) & (res.adapted40 == res.label)).sum()
print("adapted40 minus adapted20:", round((c_ - b_) / len(res), 3),
      "| discordant pairs:", b_, c_, "| rough SE:", round(np.sqrt(b_ + c_) / len(res), 3))

out = base + "raabin_fresh_test2_scores.csv"
wts = base + "resnet18_wbc_v2_raabin_fixesB_40.pt"
if os.path.exists(out) or os.path.exists(wts):
    print("files already exist, not overwriting")
else:
    res.to_csv(out, index=False)
    torch.save(m40.state_dict(), wts)
    print("saved", out, "and", wts)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda
background A: [255. 228. 203.]
adaptation images: 200 | fresh test images: 800
trained adapted-40 on 200 images
plain accuracy: 0.5 | recall: {'eosinophil': 0.72, 'lymphocyte': 0.565, 'monocyte': 0.53, 'neutrophil': 0.185}
adapted20 accuracy: 0.765 | recall: {'eosinophil': 0.81, 'lymphocyte': 0.795, 'monocyte': 0.58, 'neutrophil': 0.875}
adapted40 accuracy: 0.848 | recall: {'eosinophil': 0.775, 'lymphocyte': 0.96, 'monocyte': 0.765, 'neutrophil': 0.89}
adapted40 minus adapted20: 0.082 | discordant pairs: 27 93 | rough SE: 0.014
saved /content/drive/MyDrive/WBC-Folder/raabin_fresh_test2_scores.csv and /content/drive/MyDrive/WBC-Folder/resnet18_wbc_v2_raabin_fixesB_40.pt
